In [25]:
import torch
import torch.nn as nn
import torchvision.models as models


**Featuer Extraction**
Siamese (currently resnet18)

In [26]:

class SiameseEncoder(nn.Module):
 def __init__(self):
    super().__init__()

  
    backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    # remove avgpool + fc
    self.encoder = nn.Sequential(*list(backbone.children())[:-2])

 def forward(self, img1, img2):
    f1 = self.encoder(img1)
    f2 = self.encoder(img2)

    return f1, f2


# test

model = SiameseEncoder()

img1 = torch.randn(1, 3, 256, 256)
img2 = torch.randn(1, 3, 256, 256)

f1, f2 = model(img1, img2)

print("F1 shape:", f1.shape)
print("F2 shape:", f2.shape)




F1 shape: torch.Size([1, 512, 8, 8])
F2 shape: torch.Size([1, 512, 8, 8])


**Feature Difference**

In [27]:
# diff = torch.abs(f1 - f2)
# print("Difference shape:", diff.shape)

In [28]:
attn = nn.MultiheadAttention(embed_dim=512, num_heads=8, batch_first=True)

# reshape F1 and F2

f1_tokens = f1.flatten(2).permute(0, 2, 1)   # [1,64,512]
f2_tokens = f2.flatten(2).permute(0, 2, 1)   # [1,64,512]

# cross attention: Q from F1, K,V from F2

attn_out, _ = attn(f1_tokens, f2_tokens, f2_tokens)

print("Attention output shape:", attn_out.shape)

attn_map = attn_out.permute(0, 2, 1).reshape(1, 512, 8, 8)

print("Attention map shape:", attn_map.shape)


Attention output shape: torch.Size([1, 64, 512])
Attention map shape: torch.Size([1, 512, 8, 8])


**Decoder**
[1, 512, 8, 8]
->
[1, 256, 16, 16]

In [31]:
decoder1 = nn.Sequential(
nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2),
nn.ReLU()
)

x = decoder1(diff)

print("Decoder output shape:", x.shape)


# second decoder

decoder2 = nn.Sequential(
nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2),
nn.ReLU()
)

x = decoder2(x)

print("Decoder2 output shape:", x.shape)


decoder3 = nn.Sequential(
nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2),
nn.ReLU()
)

decoder4 = nn.Sequential(
nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),
nn.ReLU()
)

decoder5 = nn.Sequential(
nn.ConvTranspose2d(32, 1, kernel_size=2, stride=2),
nn.Sigmoid()
)

x = decoder3(x)
x = decoder4(x)
x = decoder5(x)

print("Final output shape:", x.shape)

print("Min:", x.min().item())
print("Max:", x.max().item())




Decoder output shape: torch.Size([1, 256, 16, 16])
Decoder2 output shape: torch.Size([1, 128, 32, 32])
Final output shape: torch.Size([1, 1, 256, 256])
Min: 0.4328998625278473
Max: 0.4956139922142029


Stage 1 — Dual Temporal Input
Input Image T1
Input Image T2

Stage 2 — Shared Siamese Encoder (ResNet18 Backbone)
T1 → ResNet18 Encoder → F1
T2 → ResNet18 Encoder → F2

Output:
F1 = [1,512,8,8]
F2 = [1,512,8,8]

This stage corresponds to Siamese deep feature extraction used in early Siamese change detection papers and urban Siamese frameworks

Stage 3 — Tokenization for Transformer Attention
F1 → flatten → tokens
F2 → flatten → tokens

Shape:
[1,512,8,8] → [1,64,512]

Stage 4 — Cross Attention Fusion
Q = F1
K = F2
V = F2

Cross-attention:
Attention(F1,F2)
Output:
attn_out = [1,64,512]

Stage 5 — Attention Map Reconstruction
[1,64,512] → [1,512,8,8]

Result:
attn_map

Stage 6 — Difference Generation
(Current implied stage)
diff = |F1 - attn_map|
This is your current feature difference map.
Output:
diff = [1,512,8,8]


Stage 7 — Decoder Level 1
512 → 256
8×8 → 16×16

Output:
[1,256,16,16]


Stage 8 — Decoder Level 2
256 → 128
16×16 → 32×32
Output:
[1,128,32,32]
Stage 9 — Decoder Level 3
128 → 64
32×32 → 64×64
Stage 10 — Decoder Level 4
64 → 32
64×64 → 128×128
Stage 11 — Decoder Level 5 (Final Pixel Change Map)
32 → 1
128×128 → 256×256

Final:
Change Map = [1,1,256,256]

Stage 12 — Sigmoid Binary Pixel Probability
0 = no change
1 = change
Output values:
0 to 1 probability map
Full Current Architecture in One Line
T1,T2
→ Siamese Encoder
→ Cross Attention
→ Difference Map
→ Decoder Stack
→ Pixel Change Map
Current Architecture Position Relative to Full Research Architecture

You have completed:
✅ Siamese backbone
✅ Transformer fusion
✅ Pixel decoder